# NMF Topic Model — England k=30 No Media

**Corpus:** 1,197 articles (Schools Week removed from 3,943 England training set)

**Sources:** GOV.UK, FFT, EPI, Nuffield, FED

**Model:** NMF, k=30, max_features=3000, min_df=3, init=nndsvd, max_iter=1000

**Purpose:** Source composition sensitivity test. Same k=30 and preprocessing as the production model (`nmf_eng_30`), but with Schools Week removed (~73% of articles). Schools Week is a generalist journalism outlet that covered all policy areas; the remaining five sources are specialists with distinct editorial lanes. This notebook answers: what does "English education policy discourse" look like when the dominant media voice is removed — and what does the difference tell us about specification choices in topic modelling?

**Design:** One of four England model variants (k=5, k=15, k=30, k=30 NM) trained for the contestability demonstration. The comparison between this model and the full-source k=30 model is the core specification sensitivity finding: same method, same parameters, different corpus composition, structurally different results.

**Key findings:**
- Topic stability improved from 0.94 (full-source) to 0.96 — removing the dominant source produced more reproducible topics
- Dominant weight statistics are identical across both models (min 0.021, mean 0.195, max 0.582) — the model is equally confident despite different corpora
- Three ESFA regulatory topics (11.8% of topic space) emerge as near-duplicates — administrative boilerplate fills the space Schools Week's editorial content previously occupied
- Most topics are effectively single-source: the model discovers each source's editorial agenda, not shared policy themes
- Topics that grew without Schools Week (research, leadership, mental health) were previously diluted by volume competition, not absent from the corpus
- LLM topic naming (28/30 AGREE with human names) validates interpretability but failed to detect the ESFA triplet redundancy — LLM naming cannot substitute for structural review
- Model diagnostics are necessary but not sufficient: both models pass all evaluation checks identically, yet produce different pictures of education policy


# 0. Imports 

In [1]:
import sys
sys.path.insert(0, "/workspaces/AM1_topic_modelling")

import logging
import pandas as pd
import numpy as np
from pathlib import Path

logging.basicConfig(level=logging.INFO)

from model_pipeline.training.s02_cleaning import run_cleaning
from model_pipeline.training.s03_spacy_processing import run_spacy_processing
from model_pipeline.training.s04_vectorisation import run_vectorisation, build_vectorizer
from model_pipeline.training.s05_nmf_training import run_nmf_training, get_top_words_per_topic

import logging
logging.getLogger("gensim").setLevel(logging.WARNING)

from sklearn.decomposition import NMF
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

# 1. Load England training data

In [2]:
preprocessed_path = Path("/workspaces/AM1_topic_modelling/data/training/eng_training_nm_preprocessed.parquet")

if preprocessed_path.exists():
    print("Loading preprocessed data (skipping cleaning + spaCy)...")
    df = pd.read_parquet(preprocessed_path)
    SKIP_PREPROCESSING = True
else:
    csv_path = Path("/workspaces/AM1_topic_modelling/data/training/eng_training_nm.csv")
    df = pd.read_csv(csv_path)
    SKIP_PREPROCESSING = False

print(f"Loaded: {df.shape}")
print(f"Skip preprocessing: {SKIP_PREPROCESSING}")

Loading preprocessed data (skipping cleaning + spaCy)...
Loaded: (1198, 18)
Skip preprocessing: True


# 2. Prepare text column

The Supabase schema has `title` and `text` as separate columns. The pipeline expects a combined `text` column. Also rename `article_date` → `date` to match pipeline expectations.

In [3]:
if not SKIP_PREPROCESSING:
    # Combine title + text (same as s01_data_loader)
    df["text"] = df["title"].fillna("") + "\n\n" + df["text"].fillna("")
    df["date"] = pd.to_datetime(df["article_date"], errors="coerce")
    print(f"Text combined. Sample length: {df['text'].str.len().describe()}")
else:
    print("Skipped — using preprocessed data")

Skipped — using preprocessed data


## 3. Cleaning (s02)

In [4]:
if not SKIP_PREPROCESSING:
    df = run_cleaning(df)
    print(f"After cleaning: {df.shape}")
    print(f"Empty text_clean: {(df['text_clean'].str.len() == 0).sum()}")
else:
    print("Skipped — using preprocessed data")

Skipped — using preprocessed data


# 4. spaCy processing (s03)

This takes a few minutes on ~4k articles.

In [5]:
if not SKIP_PREPROCESSING:
    df = run_spacy_processing(df)
    print(f"After spaCy: {df.shape}")
    print(f"Empty text_final: {(df['text_final'].str.len() == 0).sum()}")
    print(f"Avg tokens per doc: {df['tokens_final'].apply(len).mean():.0f}")
else:
    print("Skipped — using preprocessed data")

Skipped — using preprocessed data


In [6]:
if not SKIP_PREPROCESSING:
    df.to_parquet("/workspaces/AM1_topic_modelling/data/training/eng_training_nm_preprocessed.parquet")
    print(f"Saved preprocessed data: {df.shape}")
else:
    print("Already loaded from parquet")

Already loaded from parquet


# 5. TF-IDF vectorisation (s04)

In [7]:
# Using same params as config.yaml: min_df=3, max_df=0.85, max_features=3000, ngram_range=(1,2)
vec_out = run_vectorisation(df)
print(f"TF-IDF matrix: {vec_out.X.shape}")
print(f"Vocabulary size: {len(vec_out.feature_names)}")
print(f"Sample features: {vec_out.feature_names[:20].tolist()}")

INFO:model_pipeline.training.s04_vectorisation:Step 04 (vectorisation): starting. Input shape=(1198, 18)
INFO:model_pipeline.training.s04_vectorisation:TF-IDF shape: (1198, 3000)
INFO:model_pipeline.training.s04_vectorisation:Vectorizer params: min_df=3 max_df=0.85 max_features=3000 ngram_range=(1, 2)
INFO:model_pipeline.training.s04_vectorisation:Sample features: ['ab', 'ability', 'able', 'absence', 'absence absence', 'absence attainment', 'absence code', 'absence disadvantaged', 'absence exclusion', 'absence illness', 'absence london', 'absence persistent', 'absence pre', 'absence primary', 'absence pupil', 'absence school', 'absence secondary', 'absence session', 'absent', 'absent pupil']


TF-IDF matrix: (1198, 3000)
Vocabulary size: 3000
Sample features: ['ab', 'ability', 'able', 'absence', 'absence absence', 'absence attainment', 'absence code', 'absence disadvantaged', 'absence exclusion', 'absence illness', 'absence london', 'absence persistent', 'absence pre', 'absence primary', 'absence pupil', 'absence school', 'absence secondary', 'absence session', 'absent', 'absent pupil']


# 6. Train NMF (k=30)

In [8]:
nmf_out = run_nmf_training(vec_out.X, n_topics=30, random_state=42, init="nndsvd", max_iter=1000)
print(f"W shape: {nmf_out.W.shape}")
print(f"H shape: {nmf_out.H.shape}")
print(f"Reconstruction error: {nmf_out.reconstruction_error:.6f}")


INFO:model_pipeline.training.s05_nmf_training:Step 05 (NMF): starting. X shape=(1198, 3000)
INFO:model_pipeline.training.s05_nmf_training:Step 05 (NMF): complete.
INFO:model_pipeline.training.s05_nmf_training:W shape=(1198, 30) | H shape=(30, 3000)
INFO:model_pipeline.training.s05_nmf_training:Reconstruction error: 26.380650


W shape: (1198, 30)
H shape: (30, 3000)
Reconstruction error: 26.380650


# 7. Topic stability across random seeds

In [9]:
seeds = [42, 123, 456, 789, 1024]
H_matrices = []

for seed in seeds:
    model = NMF(n_components=30, init="nndsvda", random_state=seed, max_iter=1000)
    model.fit(vec_out.X)
    H_matrices.append(model.components_)
    print(f"Seed {seed}: recon error = {model.reconstruction_err_:.4f}")

# Compare all pairs of H matrices
pair_scores = []
for i in range(len(seeds)):
    for j in range(i + 1, len(seeds)):
        sim = cosine_similarity(H_matrices[i], H_matrices[j])
        # Best match for each topic in run i against run j
        best_matches = sim.max(axis=1).mean()
        pair_scores.append(best_matches)
        print(f"Seeds {seeds[i]} vs {seeds[j]}: avg best-match similarity = {best_matches:.4f}")

avg_stability = np.mean(pair_scores)
print(f"\nOverall topic stability: {avg_stability:.4f}")
print(f"Previous stability (old model): 0.94")
print(f"Interpretation: >0.90 = highly stable, 0.80-0.90 = acceptable, <0.80 = unstable")


Seed 42: recon error = 26.4470
Seed 123: recon error = 26.4368
Seed 456: recon error = 26.4434
Seed 789: recon error = 26.4376
Seed 1024: recon error = 26.4449
Seeds 42 vs 123: avg best-match similarity = 0.9650
Seeds 42 vs 456: avg best-match similarity = 0.9796
Seeds 42 vs 789: avg best-match similarity = 0.9450
Seeds 42 vs 1024: avg best-match similarity = 0.9833
Seeds 123 vs 456: avg best-match similarity = 0.9533
Seeds 123 vs 789: avg best-match similarity = 0.9464
Seeds 123 vs 1024: avg best-match similarity = 0.9574
Seeds 456 vs 789: avg best-match similarity = 0.9454
Seeds 456 vs 1024: avg best-match similarity = 0.9832
Seeds 789 vs 1024: avg best-match similarity = 0.9505

Overall topic stability: 0.9609
Previous stability (old model): 0.94
Interpretation: >0.90 = highly stable, 0.80-0.90 = acceptable, <0.80 = unstable


#### The NM model (Schools Week removed) achieves topic stability of 0.96, up from 0.94 with the full-source model. Reconstruction error is tightly clustered across five seeds (range: 26.437–26.447), and pairwise topic similarity ranges from 0.945 to 0.983 with no outlier pairs. Removing the single largest source improved reproducibility — a specification finding in itself. Source composition decisions have measurable downstream consequences for topic definitions, making this a key transparency point for contestable deployment.

# 8. Inspect topics — top words per topic

Compare with previous topic names to see if they still hold.

In [10]:
topics = get_top_words_per_topic(nmf_out.nmf_model, vec_out.feature_names, n_top_words=100)

# Display stopwords — these don't affect the model, just make the top words easier to read
display_stop = {
    "school", "education", "pupil", "student", "teacher", "year", "new", "work",
    "time", "say", "make", "good", "need", "use", "know", "want", "come", "take",
    "people", "government", "report", "system", "support", "include", "provide",
    "number", "change", "part", "set", "high", "low", "level", "national", "local",
    "public", "service", "also", "would", "could", "one", "two", "first", "last",
    "week", "month", "day", "told", "said", "according", "cent", "per", "per cent",
    "child", "children", "young", "staff", "area", "programme", "policy",
    "guidance", "framework", "response", "statement", "proposal", "approach",
    "review", "update", "document", "detail", "section", "datum", "figure",
    "survey", "rate", "score", "point", "proportion", "percentage",
    "organisation", "department", "committee", "institute", "foundation",
    "summit", "voice", "stakeholder", "partnership", "engagement",
    "scheme", "initiative", "pilot", "introduce", "implement", "launch",
    "office", "official", "notification", "recipient", "correspondence",
    "cookie", "banner", "subscribe", "contact", "submit", "accessibility",
    "share", "print", "visit", "site", "experience", "article", "news", "blog",
    "interesting", "fact", "previous", "current", "date", "information",
    "different", "large", "place", "individual", "view", "analysis",
    "thing", "way", "job"
}

In [11]:
topics = get_top_words_per_topic(nmf_out.nmf_model, vec_out.feature_names, n_top_words=30)

for topic_id, words in enumerate(topics):
    print(f"\nTopic {topic_id}: {', '.join(words)}")


Topic 0: late, late information, local authority, authority, academy, academy school, education, local, college local, esfa late, agency academy, html details, details education, details, skill agency, esfa, information, education late, education skill, information action, academy late, action education, authority education, agency, school college, provider html, html, education provider, late academy, late local

Topic 1: child, family, need, care, support, early, life, child school, standard, send, right, system, service, child child, attendance, government, poverty, vulnerable, school child, sure, inclusion, child young, social, school, child social, special, classroom, area, reading, social care

Topic 2: notice, warning notice, warning, regional director, dfe regional, director, regional, dfe, reason issue, urn notice, notice warning, urn, academy, judgement dfe, trust reason, relation, trust relation, trust, judgement, reason, director dfe, council, issue, issue inadequate, inad

In [17]:
TOPIC_NAMES_30_NM = {
    0: "esfa_late_notifications",
    1: "child_welfare_send",
    2: "dfe_warning_notices",
    3: "pupil_attainment_data",
    4: "post16_qualifications_equity",
    5: "skills_employer_workforce",
    6: "esfa_funding_assurance",
    7: "ofsted_inspection_reform",
    8: "ofqual_exam_regulation",
    9: "research_social_inequality",
    10: "pupil_absence_attendance",
    11: "academy_trust_governance",
    12: "teacher_workforce",
    13: "disadvantage_gap_reports",
    14: "education_system_leadership",
    15: "exclusion_suspension",
    16: "esfa_provider_regulation",
    17: "gcse_results_grades",
    18: "breakfast_clubs_wraparound",
    19: "school_operations",
    20: "accounting_officer_oversight",
    21: "youth_mental_health",
    22: "apprenticeships",
    23: "college_financial_health",
    24: "curriculum_subjects",
    25: "ai_edtech",
    26: "early_language_intervention",
    27: "rshe_parents",
    28: "childcare_early_years",
    29: "youth_justice",
}

df["dominant_topic"] = nmf_out.W.argmax(axis=1)
df["dominant_topic_name"] = df["dominant_topic"].map(TOPIC_NAMES_30_NM)
df["dominant_topic_name"].value_counts()


dominant_topic_name
pupil_attainment_data           85
education_system_leadership     77
research_social_inequality      75
youth_mental_health             69
dfe_warning_notices             67
school_operations               65
esfa_late_notifications         65
ofqual_exam_regulation          58
ofsted_inspection_reform        55
child_welfare_send              49
post16_qualifications_equity    46
curriculum_subjects             45
pupil_absence_attendance        44
teacher_workforce               43
esfa_provider_regulation        43
ai_edtech                       35
esfa_funding_assurance          34
skills_employer_workforce       32
apprenticeships                 29
academy_trust_governance        25
disadvantage_gap_reports        21
childcare_early_years           21
rshe_parents                    19
early_language_intervention     16
youth_justice                   15
breakfast_clubs_wraparound      15
exclusion_suspension            14
gcse_results_grades             12


### Topic Names and Interpretation — England k=30 NM

The 30 topics cover the breadth of English education policy discourse. Key groupings:

**Core policy areas:** child welfare & SEND (1), pupil attainment data (3), teacher workforce (12), exclusions & suspensions (15), pupil absence (10), disadvantage gap reports (13), youth mental health (21)
**Post-16 & skills:** post-16 qualifications & equity (4), skills & employer workforce (5), apprenticeships (22), college financial health (23)
**Curriculum & assessment:** GCSE results & grades (17), curriculum subjects & EBacc (24), Ofqual exam regulation (8), early language intervention (26)
**Institutional & regulatory:** Ofsted inspection reform (7), academy trust governance (11), DfE warning notices (2), accounting officer oversight (20), school operations (19)
**Current policy priorities:** AI & edtech (25), breakfast clubs & wraparound (18), childcare & early years (28), RSHE & parents (27)
**Cross-cutting:** education system leadership (14), research & social inequality (9), youth justice (29)
**Administrative/regulatory boilerplate (ESFA triplet):** provider late notifications (0), funding resource assurance (6), provider compliance regulation (16). These three are near-duplicates — k=30 over-splits ESFA content. 10% of the topic space is consumed by government compliance machinery rather than policy discourse, which is itself a finding about what dominates the English education data landscape.

### What the ESFA triplet means for contestability
The presence of three near-duplicate regulatory topics is not a model failure — it is a data composition finding. Bulk-published ESFA compliance documents were scraped alongside policy articles, news, and analysis because the corpus captures what is publicly available on English education websites. A substantial portion of that public surface is regulatory machinery, not policy debate.

This raises two contestability points:
1. **Data cleaning is a specification choice, not a neutral preprocessing step.** Filtering out compliance notices would produce different topics and different findings. The decision to retain them preserves the full picture of what these institutions publish but means the model partly describes bureaucratic process rather than policy substance. Either choice is defensible — but the choice must be disclosed.
2. **Downstream claims are shaped by what is in the corpus.** Any claim about "what English education policy discourse looks like" is affected by the inclusion of administrative content that no human reader would treat as policy commentary. A stakeholder could reasonably challenge whether compliance notices belong in the same analytical frame as Ofsted reform or child welfare. A system that does not disclose its corpus composition cannot be meaningfully contested.

# 9. Explore a topic — top articles + source concentration

In [18]:
def explore_topic(topic_id, n=5):
    """Show top N articles for a given topic, ranked by topic weight."""
    W = nmf_out.W
    topic_weights = W[:, topic_id]
    top_idx = topic_weights.argsort()[::-1][:n]

    # Topic words
    words = topics[topic_id]
    filtered = [w for w in words if w not in display_stop and len(w) > 2][:20]
    print(f"TOPIC {topic_id} ({TOPIC_NAMES_30_NM[topic_id]}) — top words: {', '.join(filtered)}")
    print(f"{'='*120}\n")

    for rank, idx in enumerate(top_idx, 1):
        row = df.iloc[idx]
        weight = topic_weights[idx]
        title = row.get("title", "No title")
        source = row.get("source", "Unknown")
        date = str(row.get("article_date", ""))[:10]
        text = row.get("text_clean", row.get("text", ""))
        if isinstance(text, str) and len(text) > 500:
            text = text[:500] + "..."

        print(f"[{rank}] weight={weight:.4f} | {source} | {date}")
        print(f"    {title}")
        print(f"    {text}\n")

# Source concentration across all topics
print("Source concentration (top 50 articles per topic):")
print("=" * 80)
for t in range(nmf_out.nmf_model.n_components):
    top_idx = nmf_out.W[:, t].argsort()[::-1][:50]
    breakdown = df.iloc[top_idx]['source'].value_counts()
    pct = (breakdown / breakdown.sum() * 100).round(0).astype(int)
    summary = ", ".join(f"{src} {p}%" for src, p in pct.items())
    print(f"Topic {t:>2} ({TOPIC_NAMES_30_NM[t]}): {summary}")

# Usage: explore_topic(0) for topic 0, explore_topic(3, n=10) for 10 articles


Source concentration (top 50 articles per topic):
Topic  0 (esfa_late_notifications): gov 100%
Topic  1 (child_welfare_send): gov 74%, fed 8%, epi 8%, nuffield 6%, fft 4%
Topic  2 (dfe_warning_notices): gov 100%
Topic  3 (pupil_attainment_data): fft 88%, epi 10%, gov 2%
Topic  4 (post16_qualifications_equity): gov 60%, epi 20%, fft 16%, nuffield 2%, fed 2%
Topic  5 (skills_employer_workforce): gov 84%, fed 10%, nuffield 6%
Topic  6 (esfa_funding_assurance): gov 100%
Topic  7 (ofsted_inspection_reform): gov 72%, fft 24%, nuffield 4%
Topic  8 (ofqual_exam_regulation): gov 100%
Topic  9 (research_social_inequality): nuffield 88%, epi 10%, fed 2%
Topic 10 (pupil_absence_attendance): fft 82%, epi 10%, gov 8%
Topic 11 (academy_trust_governance): gov 94%, fft 4%, fed 2%
Topic 12 (teacher_workforce): gov 64%, epi 14%, fed 10%, nuffield 6%, fft 6%
Topic 13 (disadvantage_gap_reports): epi 82%, nuffield 8%, gov 8%, fft 2%
Topic 14 (education_system_leadership): fed 98%, gov 2%
Topic 15 (exclusion

### Source Concentration Analysis

Source concentration (top 50 articles per topic) reveals that most topics in the NM model are effectively single-source topics:

- **GOV.UK dominates** — 100% of ESFA triplet (0, 6, 16), warning notices (2), Ofqual (8), and near-total on apprenticeships (22), trust governance (11), college financial health (23), childcare (28)
- **FFT owns the data analysis topics** — attainment data (3) at 88%, absence (10) at 82%, exclusions (15) at 66%, GCSE results (17) at 56%, curriculum subjects (24) at 56%
- **EPI owns disadvantage** — disadvantage gap reports (13) at 82%
- **Nuffield owns research/inequality** — research & social inequality (9) at 88%, youth justice (29) at 58%
- **FED owns leadership** — education system leadership (14) at 98%
- **Only a handful of topics have genuinely mixed sources** — child welfare (1), teacher workforce (12), early language (26)

#### Why this differs from the full-source (Schools Week) model

With Schools Week in the corpus (73% of all articles), topics appeared more mixed-source — but this was because one generalist journalism outlet covered everything, not because multiple sources converged on shared themes. Without Schools Week, the remaining sources are specialists: GOV.UK publishes regulation, FFT publishes data analysis, EPI publishes disadvantage research, Nuffield publishes social research. The topic structure aligns with the source structure because each source has a distinct editorial lane.

#### What this means

This is not a model failure — it is a property of the data. English education publishing is institutionally segmented. The model faithfully reflects that segmentation. But a user seeing "30 education policy topics" would assume they emerged from shared discourse. They did not — they emerged from five organisations with five different missions. This must be disclosed for any downstream claim to be contestable. For cross-jurisdictional comparison, it also means that differences in topic distributions may reflect differences in which sources were available per country, not differences in policy.


# 10. LLM-suggested topic names

Send top words per topic to Claude for naming suggestions. Compare with existing names.

In [19]:
import os
from pathlib import Path
from dotenv import load_dotenv
from anthropic import Anthropic
import json
import re

load_dotenv(Path("/workspaces/AM1_topic_modelling/.env"))
client = Anthropic(api_key=os.environ["ANTHROPIC_API_KEY"])

# Build keyword lists (without display stopwords)
topic_keyword_lines = []
for i, words in enumerate(topics):
    filtered = [w for w in words if w not in display_stop and len(w) > 2][:30]
    topic_keyword_lines.append(f"Topic {i}: {', '.join(filtered)}")

my_name_lines = []
for i in range(30):
    my_name_lines.append(f"Topic {i}: proposed name is '{TOPIC_NAMES_30_NM.get(i, 'unknown')}'")

prompt = f"""You are helping label topics from an NMF topic model trained on UK education policy documents (2023-2025).
The corpus includes articles from government departments, think tanks, media outlets, and research organisations in England.
This model has k=30 topics. The corpus has SchoolsWeek removed (no media variant).

STEP 1: For each topic below, suggest a name based ONLY on the keywords. Do not look ahead to Step 2.

{chr(10).join(topic_keyword_lines)}

For each topic return:
- suggested_name (2-4 words, snake_case)
- explanation (one sentence)

STEP 2: Now compare your suggestions against these human-proposed names for the same topics:

{chr(10).join(my_name_lines)}

For each topic, assess:
- proposed_name_assessment: "AGREE" (proposed name fits the keywords well), "CLOSE" (reasonable but could be more precise), or "DISAGREE" (proposed name doesn't capture the topic well)
- proposed_name_note: one sentence explaining why

Return ONLY a JSON list combining both steps:
[
  {{"topic": 0, "suggested_name": "name", "explanation": "why", "proposed_name_assessment": "AGREE", "proposed_name_note": "why"}},
  ...
]
No other text, no markdown, no code fences."""

response = client.messages.create(
    model="claude-sonnet-4-20250514",
    max_tokens=4096,
    messages=[{"role": "user", "content": prompt}],
)

llm_text = response.content[0].text

# Strip markdown code fences if present
cleaned = re.sub(r'^```(?:json)?\n?', '', llm_text.strip())
cleaned = re.sub(r'\n?```$', '', cleaned.strip())
llm_results = json.loads(cleaned)

# Summary table
print(f"{'Topic':>5}  {'My Name':<35}  {'Status':<8}  {'LLM Suggestion':<35}  Notes")
print("=" * 140)
for r in llm_results:
    i = r["topic"]
    mine = TOPIC_NAMES_30_NM.get(i, "???")
    print(f"{i:>5}  {mine:<35}  {r['proposed_name_assessment']:<8}  {r['suggested_name']:<35}  {r['proposed_name_note']}")

# Detailed explanations
print("\n\nDETAILED EXPLANATIONS:")
print("=" * 80)
for r in llm_results:
    print(f"\nTopic {r['topic']}: {r['suggested_name']}")
    print(f"  Why: {r['explanation']}")
    print(f"  My name '{TOPIC_NAMES_30_NM.get(r['topic'], '???')}': {r['proposed_name_assessment']} — {r['proposed_name_note']}")

# Save
with open("/workspaces/AM1_topic_modelling/data/evaluation_outputs/llm_topic_review_k30_nm.json", "w") as f:
    json.dump(llm_results, f, indent=2)
print("\nSaved to data/evaluation_outputs/llm_topic_review_k30_nm.json")


INFO:httpx:HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"


Topic  My Name                              Status    LLM Suggestion                       Notes
    0  esfa_late_notifications              AGREE     esfa_late_information                The proposed name accurately captures the ESFA late notification theme evident in the keywords.
    1  child_welfare_send                   AGREE     child_welfare_send                   The proposed name perfectly captures both the child welfare and SEND elements prominent in the keywords.
    2  dfe_warning_notices                  AGREE     dfe_warning_notices                  The proposed name precisely reflects the focus on DfE warning notices to educational institutions.
    3  pupil_attainment_data                CLOSE     pupil_attainment_measures            The proposed name captures the pupil attainment focus well, though 'measures' might be more comprehensive than just 'data'.
    4  post16_qualifications_equity         AGREE     university_qualifications_progression  The proposed name effe

### LLM Topic Name Validation

LLM-assisted naming (Claude, top 30 keywords per topic) was compared against manually assigned names across all 30 topics. **Results: 28/30 AGREE, 2/30 CLOSE, 0/30 DISAGREE.**

The two CLOSE cases are minor wording differences:
- Topic 3: `pupil_attainment_data` (manual) vs `pupil_attainment_measures` (LLM)
- Topic 12: `teacher_workforce` (manual) vs `teacher_recruitment_retention` (LLM)

Manual names retained in both cases — `pupil_attainment_data` is consistent with FFT's statistical focus, and `teacher_workforce` is broader than just recruitment/retention.

#### Contestability points

1. **High convergence validates interpretability** — topics have clear semantic identity, not statistical artefacts.
2. **The LLM did not flag the ESFA triplet.** It named Topics 0, 6, and 16 as if they were distinct. A human spotted the near-duplication immediately. LLM naming works per-topic but has no cross-topic awareness — it cannot detect redundancy, over-splitting, or structural issues in the model as a whole.
3. **Compared to other England models:** The NM k=30 model's clean separation (stability 0.96) likely produces higher LLM-human agreement than the full-source model, where Schools Week's editorial framing blurs topic boundaries. Lower-k models (k=5, k=15) would agree trivially because topics are too broad to name ambiguously.


# 11. Topic distribution — how many articles per dominant topic?

In [23]:
dominant_topics = nmf_out.W.argmax(axis=1)
dominant_weights = nmf_out.W.max(axis=1)

topic_counts = pd.Series(dominant_topics).value_counts().sort_index()
topic_df = pd.DataFrame({
    "topic_num": topic_counts.index,
    "topic_name": [TOPIC_NAMES_30_NM.get(i, "???") for i in topic_counts.index],
    "count": topic_counts.values,
    "pct": (topic_counts.values / len(dominant_topics) * 100).round(1),
})
print(topic_df.to_string(index=False))
print(f"\nDominant weight — min: {dominant_weights.min():.4f}, mean: {dominant_weights.mean():.4f}, max: {dominant_weights.max():.4f}")


 topic_num                   topic_name  count  pct
         0      esfa_late_notifications     65  5.4
         1           child_welfare_send     49  4.1
         2          dfe_warning_notices     67  5.6
         3        pupil_attainment_data     85  7.1
         4 post16_qualifications_equity     46  3.8
         5    skills_employer_workforce     32  2.7
         6       esfa_funding_assurance     34  2.8
         7     ofsted_inspection_reform     55  4.6
         8       ofqual_exam_regulation     58  4.8
         9   research_social_inequality     75  6.3
        10     pupil_absence_attendance     44  3.7
        11     academy_trust_governance     25  2.1
        12            teacher_workforce     43  3.6
        13     disadvantage_gap_reports     21  1.8
        14  education_system_leadership     77  6.4
        15         exclusion_suspension     14  1.2
        16     esfa_provider_regulation     43  3.6
        17          gcse_results_grades     12  1.0
        18  

### NM vs Full-Source Model Comparison

Dominant weight statistics are identical across both models: min 0.021, mean 0.195, max 0.582. The model is equally confident in both cases — the statistical structure of NMF at k=30 is stable across source compositions. But the content of the topics changes completely.

**Topics that grew without Schools Week** (volume competition effect — specialist sources no longer diluted by 2,890 Schools Week articles):
- `research_social_inequality`: 3.6% → 6.3% (Nuffield content clusters tightly)
- `education_system_leadership`: 1.2% → 6.4% (FED content emerges)
- `youth_mental_health`: 1.3% → 5.8% (specialist mental health content no longer absorbed)

**Topics that shrank:** `gcse_results_grades` 2.7% → 1.0% (Schools Week covered results day heavily)

**ESFA triplet takes 11.8%** of topic space (5.4% + 2.8% + 3.6%) — regulatory boilerplate fills the space Schools Week's editorial content previously occupied.

**Key finding for contestability:** Model diagnostics (reconstruction error, dominant weights, coherence) are necessary but not sufficient. Both models score identically on statistical measures but produce structurally different topic landscapes. A model can pass all evaluation checks and still produce findings that are artefacts of source composition. The only way to detect this is to perturb the corpus and compare — which is what this NM variant provides.


# 12. Topic distribution by source

Check if any source dominates a topic (specification choice: corpus composition).

In [24]:
df["dominant_topic"] = dominant_topics
df["dominant_topic_name"] = df["dominant_topic"].map(TOPIC_NAMES_30_NM)
ct = pd.crosstab(df["source"], df["dominant_topic_name"], normalize="columns").round(2)
print("Source distribution per topic (column-normalised):")
ct


Source distribution per topic (column-normalised):


dominant_topic_name,academy_trust_governance,accounting_officer_oversight,ai_edtech,apprenticeships,breakfast_clubs_wraparound,child_welfare_send,childcare_early_years,college_financial_health,curriculum_subjects,dfe_warning_notices,...,post16_qualifications_equity,pupil_absence_attendance,pupil_attainment_data,research_social_inequality,rshe_parents,school_operations,skills_employer_workforce,teacher_workforce,youth_justice,youth_mental_health
source,,,,,,,,,,,,,,,,,,,,,
epi,0.00,0.00,0.06,0.03,0.07,0.08,0.10,0.0,0.02,0.0,...,0.26,0.11,0.15,0.16,0.00,0.17,0.00,0.16,0.0,0.17
fed,0.04,0.08,0.11,0.00,0.13,0.12,0.00,0.0,0.00,0.0,...,0.04,0.00,0.00,0.01,0.11,0.05,0.12,0.12,0.0,0.07
fft,0.00,0.00,0.09,0.00,0.00,0.06,0.00,0.0,0.67,0.0,...,0.07,0.82,0.80,0.00,0.11,0.18,0.00,0.07,0.0,0.09
gov,0.96,0.92,0.66,0.97,0.80,0.69,0.86,1.0,0.31,1.0,...,0.61,0.07,0.05,0.05,0.74,0.60,0.81,0.56,0.0,0.54
nuffield,0.00,0.00,0.09,0.00,0.00,0.04,0.05,0.0,0.00,0.0,...,0.02,0.00,0.00,0.77,0.05,0.00,0.06,0.09,1.0,0.13


### Source-Topic Crosstab: Single-Source vs Shared Discourse

Three topics are 100% single-source: `youth_justice` (Nuffield), `college_financial_health` (GOV.UK), `dfe_warning_notices` (GOV.UK). FFT dominates data topics: `pupil_attainment_data` 80%, `pupil_absence_attendance` 82%, `curriculum_subjects` 67%.

**Genuinely multi-source topics** (no source >70%) represent shared discourse rather than one organisation's output:
- `ai_edtech` — gov 66%, fed 11%, fft 9%, nuffield 9%, epi 6%
- `teacher_workforce` — gov 56%, epi 16%, fed 12%, nuffield 9%, fft 7%
- `school_operations` — gov 60%, fft 18%, epi 17%
- `youth_mental_health` — gov 54%, epi 17%, nuffield 13%, fft 9%, fed 7%

These multi-source topics are the strongest candidates for cross-jurisdictional comparison — they capture genuine thematic convergence. Single-source topics are better understood as claims about what that source publishes, not about the policy landscape as a whole.


# 13. Save retrained model artifacts

In [27]:
import json
from datetime import datetime

run_id = datetime.now().strftime("%Y-%m-%d_%H%M%S")
run_dir = Path(f"/workspaces/AM1_topic_modelling/experiments/outputs/runs/{run_id}")
run_dir.mkdir(parents=True, exist_ok=True)

# Topic summary for sensitivity page
topic_summary = {
    "model_id": "nmf_eng_30_nm",
    "n_topics": 30,
    "n_articles": len(df),
    "topics": [
        {"topic_num": i, "name": TOPIC_NAMES_30_NM[i], "count": int((nmf_out.W.argmax(axis=1) == i).sum())}
        for i in range(30)
    ]
}
with open(run_dir / "topic_summary.json", "w") as f:
    json.dump(topic_summary, f, indent=2)

# Topic names
with open(run_dir / "topic_names.json", "w") as f:
    json.dump(TOPIC_NAMES_30_NM, f, indent=2)

# Run metadata
metadata = {
    "run_id": run_id,
    "model_id": "nmf_eng_30_nm",
    "n_articles": len(df),
    "n_topics": 30,
    "init": "nndsvd",
    "random_state": 42,
    "max_iter": 1000,
    "reconstruction_error": float(nmf_out.reconstruction_error),
    "corpus": "eng_training_3943_supabase",
    "note": "England k=30 for contestability comparison. Not a production model."
}
with open(run_dir / "run_metadata.json", "w") as f:
    json.dump(metadata, f, indent=2)

print(f"Saved to {run_dir}")


Saved to /workspaces/AM1_topic_modelling/experiments/outputs/runs/2026-03-24_174315


# 15. Save dashboard summary JSON (for Specification Sensitivity page)

In [26]:
import json

dominant = nmf_out.W.argmax(axis=1)

topic_data = []
for i in range(30):
    mask = dominant == i
    count = int(mask.sum())
    source_counts = df.loc[mask, "source"].value_counts()
    top_source = source_counts.index[0] if len(source_counts) > 0 else "unknown"
    top_source_pct = round(float(source_counts.iloc[0] / source_counts.sum()), 2) if len(source_counts) > 0 else 0

    topic_data.append({
        "topic_num": i,
        "name": TOPIC_NAMES_30_NM[i],
        "count": count,
        "pct": round(count / len(df) * 100, 1),
        "top_source": top_source,
        "top_source_pct": top_source_pct,
        "single_source": top_source_pct > 0.90,
    })

summary = {
    "model_id": "nmf_eng_30_nm",
    "n_topics": 30,
    "n_articles": len(df),
    "metrics": {
        "reconstruction_error": round(float(nmf_out.reconstruction_error), 4),
        "stability": round(float(avg_stability), 4),
        "mean_dominant_weight": round(float(nmf_out.W.max(axis=1).mean()), 4),
        "max_dominant_weight": round(float(nmf_out.W.max(axis=1).max()), 4),
    },
    "topics": topic_data,
}

out_path = "/workspaces/AM1_topic_modelling/data/evaluation_outputs/nmf_eng_30_nm_summary.json"
with open(out_path, "w") as f:
    json.dump(summary, f, indent=2)

print(f"Saved to {out_path}")
print(f"Single-source topics: {sum(1 for t in topic_data if t['single_source'])}/30")


Saved to /workspaces/AM1_topic_modelling/data/evaluation_outputs/nmf_eng_30_nm_summary.json
Single-source topics: 11/30


## Final Summary: England k=30 No Media — Source Composition Sensitivity

This notebook trained an NMF topic model (k=30) on the England corpus with Schools Week removed (1,197 articles from five specialist sources). Same preprocessing, same parameters, same algorithm as the full-source production model — only the corpus composition changed.

### What we found

**1. The model is more stable without Schools Week.** Topic stability rose from 0.94 to 0.96. Reconstruction error is tightly clustered across five seeds (range: 26.437–26.447). Removing the single largest source reduced seed-dependent noise in topic definitions.

**2. The statistical properties are identical; the substantive findings are not.** Both models produce the same dominant weight statistics (min 0.021, mean 0.195, max 0.582). A model evaluator looking only at diagnostics would see two equally good models. But the topics are structurally different — the largest topic shifts from `teacher_pay` (journalism-driven) to `pupil_attainment_data` (data analysis-driven). What counts as "the education debate" depends entirely on who is in the corpus.

**3. Topics are largely single-source.** Without Schools Week's connective tissue, the topic structure mirrors the source structure. GOV.UK owns regulation, FFT owns data analysis, EPI owns disadvantage, Nuffield owns research, FED owns leadership. The model discovers editorial agendas, not a shared policy discourse. The few genuinely multi-source topics (AI/edtech, teacher workforce, youth mental health) are the strongest candidates for cross-jurisdictional analysis.

**4. The ESFA triplet reveals data composition.** Three near-duplicate topics (11.8% of topic space) capture government regulatory boilerplate — compliance notices that no human would read as policy commentary. Their presence is a specification choice: retaining them preserves the full picture of what institutions publish, but means the model partly describes bureaucratic process rather than policy substance.

**5. LLM naming validates topics but misses structure.** 28/30 LLM-suggested names agreed with human names, confirming topic interpretability. But the LLM reviewed each topic independently and did not flag the ESFA triplet as redundant. Per-topic LLM naming cannot substitute for human structural review.

### What this means for contestability

This notebook demonstrates specification sensitivity in its purest form. The method is constant — only the input changes. The model passes all diagnostic checks in both configurations. Yet the findings are different enough to support different policy conclusions. Any system that presents topic model outputs without disclosing corpus composition, source concentration, and the effect of source inclusion/exclusion decisions is presenting an artefact as a finding. The comparison between this model and the full-source model is not about which is correct — it is evidence that the choice is consequential and must be made visible.

### Artefacts saved
- `experiments/outputs/runs/{run_id}/` — model, vectoriser, topic names, metadata
- `data/evaluation_outputs/nmf_eng_30_nm_summary.json` — dashboard summary with per-topic source concentration
- `data/evaluation_outputs/llm_topic_review_k30_nm.json` — LLM naming comparison
